# Resale Flat Price — Feature Engineering

Builds on the cleaned dataset (`resale_flat_price_cleaned.csv`) and enriches it with
geospatial, demographic, and temporal features from the project's raw datasets.

**Feature groups engineered:**
1. Geospatial — lat/lng from `hdb.csv`, region, planning area
2. MRT proximity — distance to nearest MRT station
3. POI density — counts / distances to nearby points of interest
4. Transport mode shares — from OneMap transport-to-work data
5. Dwelling mix — from OneMap dwelling-type data
6. Temporal — quarter, year-month index, price trend lag
7. Interactions & transforms — log price, lease age², floor×storey

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.4f}".format)

In [2]:
# Load cleaned resale data
df = pd.read_csv("../dataset/processed/resale_flat_price_cleaned.csv", parse_dates=["month"])
print(f"Resale transactions: {len(df):,} rows, {df.shape[1]} cols")
df.head(2)

Resale transactions: 223,208 rows, 15 cols


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,year,remaining_lease_years,price_per_sqm,storey_mid
0,2017-01-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0000,Improved,1979,61 years 04 months,"232,000.0000",2017,61.3300,"5,272.7273",11.0000
1,2017-01-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0000,New Generation,1978,60 years 07 months,"250,000.0000",2017,60.5800,"3,731.3433",2.0000


In [3]:
# Load raw reference datasets
hdb = pd.read_csv("../dataset/raw/hdb.csv")
mrt = pd.read_csv("../dataset/raw/mrt.csv")
poi = pd.read_csv("../dataset/raw/poi.csv")
transport_work = pd.read_csv("../dataset/raw/onemap_transport_to_work.csv")
dwelling = pd.read_csv("../dataset/raw/onemap_dwelling.csv")

print(f"HDB buildings: {len(hdb):,}")
print(f"MRT stations:  {len(mrt):,}")
print(f"POI:           {len(poi):,}")
print(f"Transport:     {len(transport_work):,}")
print(f"Dwelling:      {len(dwelling):,}")

HDB buildings: 12,442
MRT stations:  247
POI:           8,672
Transport:     219
Dwelling:      219


---
## 1 · Geospatial Features — Join with HDB building data

Join resale transactions to `hdb.csv` on (`block`, `street_name`) to obtain
latitude, longitude, planning area, and region for each transaction.

In [4]:
# Normalise join keys — uppercase, strip whitespace
df["block_key"] = df["block"].astype(str).str.strip().str.upper()
df["street_key"] = df["street_name"].astype(str).str.strip().str.upper()

hdb["block_key"] = hdb["blk_no"].astype(str).str.strip().str.upper()
hdb["street_key"] = hdb["street"].astype(str).str.strip().str.upper()

# Select columns to bring in from hdb
hdb_cols = ["block_key", "street_key", "lat", "lng",
            "PLN_AREA_N", "REGION_N", "max_floor_lvl", "year_completed",
            "total_dwelling_units", "market_hawker", "multistorey_carpark"]
hdb_dedup = hdb.loc[:, hdb_cols].drop_duplicates(subset=["block_key", "street_key"])

df = df.merge(hdb_dedup, on=["block_key", "street_key"], how="left")

matched = df["lat"].notna().sum()
print(f"Geo-matched: {matched:,} / {len(df):,} ({matched/len(df)*100:.1f}%)")
df[["block", "street_name", "lat", "lng", "PLN_AREA_N", "REGION_N"]].head()

Geo-matched: 223,208 / 223,208 (100.0%)


,block,street_name,lat,lng,PLN_AREA_N,REGION_N
0,406,ANG MO KIO AVE 10,1.3620,103.8539,ANG MO KIO,NORTH-EAST REGION
1,108,ANG MO KIO AVE 4,1.3709,103.8380,ANG MO KIO,NORTH-EAST REGION
2,118,ANG MO KIO AVE 4,1.3733,103.8355,ANG MO KIO,NORTH-EAST REGION
3,119,ANG MO KIO AVE 3,1.3696,103.8447,ANG MO KIO,NORTH-EAST REGION
4,150,ANG MO KIO AVE 5,1.3768,103.8420,ANG MO KIO,NORTH-EAST REGION


In [5]:
# Building-level features from hdb.csv
df["max_floor_lvl"] = pd.to_numeric(df["max_floor_lvl"], errors="coerce")
df["year_completed"] = pd.to_numeric(df["year_completed"], errors="coerce")
df["total_dwelling_units"] = pd.to_numeric(df["total_dwelling_units"], errors="coerce")

# Building age at time of transaction
df["building_age"] = df["year"] - df["year_completed"]

# Boolean amenity flags
df["has_market_hawker"] = (df["market_hawker"].astype(str).str.upper() == "Y").astype(int)
df["has_multistorey_carpark"] = (df["multistorey_carpark"].astype(str).str.upper() == "Y").astype(int)

print("Building features sample:")
df[["block", "street_name", "building_age", "max_floor_lvl",
    "total_dwelling_units", "has_market_hawker", "has_multistorey_carpark"]].head()

Building features sample:


,block,street_name,building_age,max_floor_lvl,total_dwelling_units,has_market_hawker,has_multistorey_carpark
0,406,ANG MO KIO AVE 10,39,12,220,0,0
1,108,ANG MO KIO AVE 4,39,12,122,0,0
2,118,ANG MO KIO AVE 4,40,12,151,0,0
3,119,ANG MO KIO AVE 3,40,11,196,0,0
4,150,ANG MO KIO AVE 5,37,12,128,0,0


---
## 2 · MRT Proximity

Use BallTree (haversine) to compute distance from each HDB block to the nearest MRT station.
Higher-floor premium flats near MRT stations command significantly higher prices.

In [6]:
from sklearn.neighbors import BallTree

R_EARTH_M = 6_371_000

# Prepare MRT coordinates
mrt_coords = mrt[["lat", "lng"]].dropna()
mrt_rad = np.radians(mrt_coords[["lat", "lng"]].to_numpy())
mrt_tree = BallTree(mrt_rad, metric="haversine")

# Compute distance for each unique (block, street) to avoid redundant queries
block_coords = (
    df[["block_key", "street_key", "lat", "lng"]]
    .dropna(subset=["lat", "lng"])
    .drop_duplicates(subset=["block_key", "street_key"])
)
block_rad = np.radians(block_coords[["lat", "lng"]].to_numpy())

dist_rad, idx = mrt_tree.query(block_rad, k=1)
block_coords = block_coords.copy()
block_coords["dist_to_nearest_mrt_m"] = np.asarray(dist_rad).ravel() * R_EARTH_M
block_coords["nearest_mrt"] = mrt["name"].iloc[np.asarray(idx).ravel()].values

# Merge back to full df
df = df.merge(
    block_coords[["block_key", "street_key", "dist_to_nearest_mrt_m", "nearest_mrt"]],
    on=["block_key", "street_key"],
    how="left",
)

print(f"MRT distance — median: {df['dist_to_nearest_mrt_m'].median():,.0f}m, "
      f"max: {df['dist_to_nearest_mrt_m'].max():,.0f}m")
df[["block", "street_name", "dist_to_nearest_mrt_m", "nearest_mrt"]].head()

MRT distance — median: 547m, max: 3,535m


,block,street_name,dist_to_nearest_mrt_m,nearest_mrt
0,406,ANG MO KIO AVE 10,926.8979,ANG MO KIO MRT STATION
1,108,ANG MO KIO AVE 4,"1,341.0941",ANG MO KIO MRT STATION
2,118,ANG MO KIO AVE 4,"1,411.9236",YIO CHU KANG MRT STATION
3,119,ANG MO KIO AVE 3,582.3341,ANG MO KIO MRT STATION
4,150,ANG MO KIO AVE 5,639.4552,YIO CHU KANG MRT STATION


---
## 3 · POI Proximity & Density

Compute distance to the nearest POI of key categories (schools, malls, hawker/food, parks)
and count how many POIs fall within a 1 km radius of each block.

In [7]:
# Define POI category groups based on boolean columns in poi.csv
poi_categories = {
    "school": ["school", "primary_school", "secondary_school"],
    "mall": ["shopping_mall"],
    "food": ["restaurant", "food", "cafe", "meal_takeaway"],
    "park": ["park"],
    "supermarket": ["supermarket", "grocery_or_supermarket"],
}

poi_clean = poi.dropna(subset=["lat", "lng"]).copy()
poi_clean["lat"] = pd.to_numeric(poi_clean["lat"], errors="coerce")
poi_clean["lng"] = pd.to_numeric(poi_clean["lng"], errors="coerce")
poi_clean = poi_clean.dropna(subset=["lat", "lng"])

# Reuse block_coords from MRT section
block_rad = np.radians(
    block_coords[["lat", "lng"]].to_numpy()
)

poi_new_cols: list[str] = []

for cat_name, cat_cols in poi_categories.items():
    # Filter POIs that match any column in the category
    valid_cols = [c for c in cat_cols if c in poi_clean.columns]
    if not valid_cols:
        continue
    mask = poi_clean[valid_cols].apply(
        lambda col: col.astype(str).str.upper().isin(["TRUE", "1", "Y"]),
    ).any(axis=1)
    cat_pois = poi_clean.loc[mask, ["lat", "lng"]]
    if cat_pois.empty:
        continue

    cat_rad = np.radians(cat_pois[["lat", "lng"]].to_numpy())
    tree = BallTree(cat_rad, metric="haversine")

    # Nearest distance
    dist, _ = tree.query(block_rad, k=1)
    col_dist = f"dist_to_{cat_name}_m"
    block_coords[col_dist] = np.asarray(dist).ravel() * R_EARTH_M
    poi_new_cols.append(col_dist)

    # Count within 1 km
    radius_rad = 1000 / R_EARTH_M
    counts = tree.query_radius(block_rad, r=radius_rad, count_only=True)
    col_count = f"n_{cat_name}_within_1km"
    block_coords[col_count] = np.asarray(counts)
    poi_new_cols.append(col_count)

print("POI features per block:")
block_coords[poi_new_cols].describe()

POI features per block:


,dist_to_school_m,n_school_within_1km,dist_to_mall_m,n_mall_within_1km,dist_to_food_m,n_food_within_1km,dist_to_park_m,n_park_within_1km,dist_to_supermarket_m,n_supermarket_within_1km
count,"9,693.0000","9,693.0000","9,693.0000","9,693.0000","9,693.0000","9,693.0000","9,693.0000","9,693.0000","9,693.0000","9,693.0000"
mean,382.9768,4.3666,526.6325,3.0485,233.9187,23.8305,"1,184.2015",0.5295,352.0244,4.6175
std,244.5931,1.9476,288.1840,2.8124,135.4531,14.1002,572.5935,0.7655,200.5659,1.9167
min,2.5026,0.0000,6.0086,0.0000,1.1923,0.0000,44.6501,0.0000,1.1923,0.0000
25%,221.4025,3.0000,308.3779,1.0000,135.7540,14.0000,749.2575,0.0000,210.9582,3.0000
50%,335.4480,4.0000,472.9888,2.0000,211.1777,22.0000,"1,154.5163",0.0000,323.1107,5.0000
75%,477.4542,5.0000,705.7419,4.0000,307.7244,31.0000,"1,575.3557",1.0000,450.6321,6.0000
max,"3,324.0809",13.0000,"3,222.3442",36.0000,"1,281.7028",160.0000,"3,528.0413",5.0000,"3,201.0468",11.0000


In [8]:
# Merge POI features back to main df
poi_merge_cols = ["block_key", "street_key"] + poi_new_cols

# Debug: confirm columns exist on block_coords before merging
print("POI cols on block_coords:", [c for c in poi_new_cols if c in block_coords.columns])
print("POI cols missing:        ", [c for c in poi_new_cols if c not in block_coords.columns])

df = df.merge(block_coords[poi_merge_cols], on=["block_key", "street_key"], how="left")

# Only display columns that actually made it into df
display_cols = [c for c in poi_new_cols if c in df.columns]
print(f"\nPOI features merged ({len(display_cols)} cols). Sample:")
df[["block", "street_name"] + display_cols].head()

POI cols on block_coords: ['dist_to_school_m', 'n_school_within_1km', 'dist_to_mall_m', 'n_mall_within_1km', 'dist_to_food_m', 'n_food_within_1km', 'dist_to_park_m', 'n_park_within_1km', 'dist_to_supermarket_m', 'n_supermarket_within_1km']
POI cols missing:         []

POI features merged (10 cols). Sample:


,block,street_name,dist_to_school_m,n_school_within_1km,dist_to_mall_m,n_mall_within_1km,dist_to_food_m,n_food_within_1km,dist_to_park_m,n_park_within_1km,dist_to_supermarket_m,n_supermarket_within_1km
0,406,ANG MO KIO AVE 10,285.5361,3,"1,032.4408",0,159.0495,15,702.3726,1,298.9434,3
1,108,ANG MO KIO AVE 4,400.9247,5,899.0406,1,255.8124,13,637.7739,1,388.0553,2
2,118,ANG MO KIO AVE 4,328.6512,4,"1,179.4009",0,88.0320,11,853.2234,1,461.7497,1
3,119,ANG MO KIO AVE 3,346.3735,5,302.1230,4,190.1357,34,630.5050,3,334.9699,5
4,150,ANG MO KIO AVE 5,170.4295,4,698.3152,3,108.5067,17,"1,137.0965",0,410.4652,2


---
## 4 · Planning-Area Demographics — Transport & Dwelling

Join OneMap planning-area-level data to add neighbourhood-level context.
Use the latest available year per planning area (census years: 2000, 2010, 2015, 2020).

In [9]:
# --- Transport to work: compute mode shares ---
tw = transport_work.copy()
tw["planning_area"] = tw["planning_area"].str.strip().str.upper()

# Use latest year per planning area
tw = tw.sort_values("year").drop_duplicates(subset=["planning_area"], keep="last")

mode_cols = [c for c in tw.columns if c not in ("planning_area", "year")]
tw[mode_cols] = tw[mode_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
tw["total_commuters"] = tw[mode_cols].sum(axis=1)

# Compute shares for key modes
tw["share_public_transport"] = (
    tw[["bus", "mrt", "mrt_bus"]].sum(axis=1)
    .div(tw["total_commuters"])
    .replace([np.inf, -np.inf], np.nan)
)
tw["share_car"] = (
    tw["car"]
    .div(tw["total_commuters"])
    .replace([np.inf, -np.inf], np.nan)
)

transport_features = tw[["planning_area", "share_public_transport", "share_car"]].copy()

print(f"Transport features for {len(transport_features)} planning areas")
transport_features.head()

Transport features for 56 planning areas


,planning_area,share_public_transport,share_car
17,BOON LAY/PIONEER,NaN,NaN
202,TENGAH,NaN,NaN
201,MANDAI,NaN,NaN
200,KALLANG,0.1554,0.1750
199,CHOA CHU KANG,0.0784,0.2038


In [10]:
# --- Dwelling type: compute HDB share vs private share ---
dw = dwelling.copy()
dw["planning_area"] = dw["planning_area"].str.strip().str.upper()
dw = dw.sort_values("year").drop_duplicates(subset=["planning_area"], keep="last")

dwell_cols = [c for c in dw.columns if c not in ("planning_area", "year")]
dw[dwell_cols] = dw[dwell_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
dw["total_dwellings"] = dw[dwell_cols].sum(axis=1)

hdb_dwell_cols = [c for c in dwell_cols if c.startswith("hdb")]
dw["share_hdb"] = (
    dw[hdb_dwell_cols].sum(axis=1)
    .div(dw["total_dwellings"])
    .replace([np.inf, -np.inf], np.nan)
)
dw["share_private"] = (
    dw[["condominiums_and_other_apartments", "landed_properties"]].sum(axis=1)
    .div(dw["total_dwellings"])
    .replace([np.inf, -np.inf], np.nan)
)

dwelling_features = dw[["planning_area", "share_hdb", "share_private"]].copy()

print(f"Dwelling features for {len(dwelling_features)} planning areas")
dwelling_features.head()

Dwelling features for 56 planning areas


,planning_area,share_hdb,share_private
17,BOON LAY/PIONEER,NaN,NaN
202,TENGAH,NaN,NaN
201,MANDAI,NaN,NaN
200,KALLANG,0.7900,0.2013
199,CHOA CHU KANG,0.8620,0.1379


In [11]:
# Merge transport & dwelling features via planning area
df["planning_area"] = df["PLN_AREA_N"].str.strip().str.upper()

df = df.merge(transport_features, on="planning_area", how="left")
df = df.merge(dwelling_features, on="planning_area", how="left")

matched_t = df["share_public_transport"].notna().sum()
matched_d = df["share_hdb"].notna().sum()
print(f"Transport matched: {matched_t:,}/{len(df):,} ({matched_t/len(df)*100:.1f}%)")
print(f"Dwelling matched:  {matched_d:,}/{len(df):,} ({matched_d/len(df)*100:.1f}%)")

Transport matched: 222,492/223,208 (99.7%)
Dwelling matched:  222,492/223,208 (99.7%)


---
## 5 · Temporal Features

Extract quarter, a continuous month index (months since first transaction), and
a rolling town-level price trend to capture local market momentum.

In [12]:
# Quarter and month index
df["quarter"] = df["month"].dt.quarter
min_month = df["month"].min()
df["month_index"] = (
    (df["month"].dt.year - min_month.year) * 12
    + (df["month"].dt.month - min_month.month)
)

# Rolling 6-month median price per town (lagged by 1 month to avoid leakage)
df = df.sort_values(["town", "month"])
town_monthly_median = (
    df.groupby(["town", "month"])["resale_price"]
    .median()
    .reset_index()
    .rename(columns={"resale_price": "town_median_price"})
)
town_monthly_median = town_monthly_median.sort_values(["town", "month"])
town_monthly_median["town_price_trend_6m"] = (
    town_monthly_median
    .groupby("town")["town_median_price"]
    .transform(lambda x: x.rolling(6, min_periods=1).mean().shift(1))
)

df = df.merge(town_monthly_median[["town", "month", "town_price_trend_6m"]],
              on=["town", "month"], how="left")

print("Temporal features sample:")
df[["month", "quarter", "month_index", "town", "town_price_trend_6m"]].head(10)

Temporal features sample:


,month,quarter,month_index,town,town_price_trend_6m
0,2017-01-01,1,0,ANG MO KIO,NaN
1,2017-01-01,1,0,ANG MO KIO,NaN
2,2017-01-01,1,0,ANG MO KIO,NaN
3,2017-01-01,1,0,ANG MO KIO,NaN
4,2017-01-01,1,0,ANG MO KIO,NaN
5,2017-01-01,1,0,ANG MO KIO,NaN
6,2017-01-01,1,0,ANG MO KIO,NaN
7,2017-01-01,1,0,ANG MO KIO,NaN
8,2017-01-01,1,0,ANG MO KIO,NaN
9,2017-01-01,1,0,ANG MO KIO,NaN


---
## 6 · Interaction & Transformed Features

Create non-linear and interaction terms motivated by EDA findings:
- **Log price** — right-skewed target benefits from log transform
- **Lease age squared** — lease decay accelerates below ~60 years
- **Floor × storey** — larger units on higher floors compound the premium

In [13]:
# Log-transformed target
df["log_resale_price"] = np.log1p(df["resale_price"])

# Lease age (how old the lease is, not remaining)
df["lease_age"] = 99 - df["remaining_lease_years"]
df["lease_age_sq"] = df["lease_age"] ** 2

# Interaction: floor area × storey midpoint
df["floor_area_x_storey"] = df["floor_area_sqm"] * df["storey_mid"]

# Storey relative to building max — how high up within its building
df["storey_ratio"] = df["storey_mid"] / df["max_floor_lvl"]

print("Transformed features sample:")
df[["resale_price", "log_resale_price", "lease_age", "lease_age_sq",
    "floor_area_x_storey", "storey_ratio"]].describe()

Transformed features sample:


,resale_price,log_resale_price,lease_age,lease_age_sq,floor_area_x_storey,storey_ratio
count,"223,208.0000","223,208.0000","223,208.0000","223,208.0000","223,208.0000","223,208.0000"
mean,"528,321.5707",13.1169,24.9183,822.1310,847.5421,0.5562
std,"188,568.5628",0.3479,14.1849,733.6371,610.7948,0.2817
min,"140,000.0000",11.8494,1.2500,1.5625,62.0000,0.0417
25%,"388,000.0000",12.8688,10.8300,117.2889,420.0000,0.3333
50%,"498,000.0000",13.1184,25.1700,633.5289,736.0000,0.5333
75%,"635,000.0000",13.3614,36.6700,"1,344.6889","1,144.0000",0.8000
max,"1,700,000.0000",14.3461,59.2500,"3,510.5625","5,311.0000",1.2500


---
## 7 · Clean Up & Export

In [14]:
df: pd.DataFrame = df  # type: ignore[has-type]

# Drop intermediate join keys and raw columns already represented by engineered features
drop_cols = ["block_key", "street_key", "market_hawker", "multistorey_carpark",
             "year_completed", "remaining_lease"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Rename for clarity
df = df.rename(columns={
    "lat": "latitude",
    "lng": "longitude",
    "PLN_AREA_N": "planning_area",
    "REGION_N": "region",
})

# Drop duplicate planning_area column if merge created one
df = df.loc[:, ~df.columns.duplicated()]

print(f"Final shape: {df.shape}")
print(f"\nNull counts (non-zero only):")
nulls = df.isnull().sum()
print(nulls[nulls > 0])
print(f"\nColumns ({df.shape[1]}):")
print(df.columns.tolist())

Final shape: (223208, 47)

Null counts (non-zero only):
share_public_transport     716
share_car                  716
share_hdb                  716
share_private              716
town_price_trend_6m       1160
dtype: int64

Columns (47):
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', 'year', 'remaining_lease_years', 'price_per_sqm', 'storey_mid', 'latitude', 'longitude', 'planning_area', 'region', 'max_floor_lvl', 'total_dwelling_units', 'building_age', 'has_market_hawker', 'has_multistorey_carpark', 'dist_to_nearest_mrt_m', 'nearest_mrt', 'dist_to_school_m', 'n_school_within_1km', 'dist_to_mall_m', 'n_mall_within_1km', 'dist_to_food_m', 'n_food_within_1km', 'dist_to_park_m', 'n_park_within_1km', 'dist_to_supermarket_m', 'n_supermarket_within_1km', 'share_public_transport', 'share_car', 'share_hdb', 'share_private', 'quarter', 'month_index', 'town_price_trend_6m', 'log_resale_price', 'lease_

In [15]:
# Quick correlation check of new features against resale_price
new_features = [
    "building_age", "max_floor_lvl", "total_dwelling_units",
    "has_market_hawker", "has_multistorey_carpark",
    "dist_to_nearest_mrt_m",
    "dist_to_school_m", "n_school_within_1km",
    "dist_to_mall_m", "n_mall_within_1km",
    "dist_to_food_m", "n_food_within_1km",
    "dist_to_park_m", "n_park_within_1km",
    "dist_to_supermarket_m", "n_supermarket_within_1km",
    "share_public_transport", "share_car",
    "share_hdb", "share_private",
    "month_index", "town_price_trend_6m",
    "lease_age", "lease_age_sq",
    "floor_area_x_storey", "storey_ratio",
]
existing = [c for c in new_features if c in df.columns]
corr_with_price = df[existing + ["resale_price"]].corr()["resale_price"].drop("resale_price").sort_values()
print("Correlation with resale_price (new features):")
print(corr_with_price.to_string())

Correlation with resale_price (new features):
lease_age_sq               -0.3165
building_age               -0.3023
lease_age                  -0.3004
dist_to_mall_m             -0.1089
dist_to_nearest_mrt_m      -0.0903
share_hdb                  -0.0832
dist_to_supermarket_m      -0.0732
total_dwelling_units       -0.0625
dist_to_park_m             -0.0606
n_school_within_1km        -0.0184
dist_to_food_m             -0.0140
has_market_hawker          -0.0079
has_multistorey_carpark    -0.0057
dist_to_school_m            0.0129
n_supermarket_within_1km    0.0486
storey_ratio                0.0502
n_park_within_1km           0.0634
share_public_transport      0.0756
share_private               0.0828
n_mall_within_1km           0.0849
n_food_within_1km           0.0904
share_car                   0.0927
month_index                 0.4023
max_floor_lvl               0.4601
floor_area_x_storey         0.4995
town_price_trend_6m         0.5107


In [16]:
# Save feature-engineered dataset
output_path = "../dataset/processed/resale_flat_price_features.csv"
df.to_csv(output_path, index=False)
print(f"Saved {len(df):,} rows × {df.shape[1]} cols to {output_path}")

Saved 223,208 rows × 47 cols to ../dataset/processed/resale_flat_price_features.csv


---
## Summary of Engineered Features

| Feature | Source | Description |
|---------|--------|-------------|
| `latitude`, `longitude` | `hdb.csv` | Block coordinates (WGS84) |
| `building_age` | `hdb.csv` + resale `year` | Transaction year − year completed |
| `max_floor_lvl` | `hdb.csv` | Maximum floor level of building |
| `total_dwelling_units` | `hdb.csv` | Total units in the building |
| `has_market_hawker` | `hdb.csv` | Whether block has market/hawker centre |
| `has_multistorey_carpark` | `hdb.csv` | Whether block has multistorey carpark |
| `dist_to_nearest_mrt_m` | `mrt.csv` | Haversine distance to nearest MRT (metres) |
| `nearest_mrt` | `mrt.csv` | Name of nearest MRT station |
| `dist_to_{cat}_m` | `poi.csv` | Distance to nearest school/mall/food/park/supermarket |
| `n_{cat}_within_1km` | `poi.csv` | Count of POIs within 1 km radius |
| `share_public_transport` | OneMap transport | Planning-area share of bus/MRT commuters |
| `share_car` | OneMap transport | Planning-area share of car commuters |
| `share_hdb` | OneMap dwelling | Planning-area share of HDB dwellings |
| `share_private` | OneMap dwelling | Planning-area share of private dwellings |
| `quarter` | resale `month` | Quarter of transaction (1–4) |
| `month_index` | resale `month` | Months since first transaction (continuous) |
| `town_price_trend_6m` | resale data | Lagged 6-month rolling median price for the town |
| `log_resale_price` | resale `resale_price` | log(1 + resale_price) |
| `lease_age` | `remaining_lease_years` | 99 − remaining lease years |
| `lease_age_sq` | `lease_age` | lease_age² (captures non-linear decay) |
| `floor_area_x_storey` | resale data | floor_area_sqm × storey_mid |
| `storey_ratio` | resale + `hdb.csv` | storey_mid / max_floor_lvl (relative height in building) |